# Unit 4 complete: multi-agent systems and collaboration

This one notebook covers the whole unit, start to finish, with small examples.
You can use it to revise, or to learn the unit from the beginning.

Our example all the way through: a **customer support ticket** (a message from an unhappy customer).

| Part | Idea | Everyday picture |
|---|---|---|
| 1 | **What is a multi-agent system?** | One person doing everything vs a team |
| 2 | **Role-based agents** | Every team member has one job title |
| 3 | **Fan-out and fan-in** | One exam, three graders, marks added when all finish |
| 4 | **Routing** | A hospital receptionist sends you to one doctor |
| 5 | **Manager and delegation** | A shop manager picks the right worker |
| 6 | **Blackboard** | Everyone writes notes on one shared whiteboard |
| 7 | **Inter-agent communication** | Three ways agents talk to each other |
| 8 | **Planner and executor** | One person writes a to-do list, others tick it off |
| 9 | **Collaborative agents** | A writer and an editor, and a team vote |
| 10 | **Distributed problem solving** | Split a big pile of work between many people |
| 11 | **Scaling** | A busy restaurant with only so many tables |
| 12 | **Workflow optimisation** | Measure first, then fix the biggest cost |
| 13 | **The full enterprise workflow** | Everything working together in one system |

**Before you start:** put `GROQ_API_KEY=...` in the project's `.env` file
(get a key at [console.groq.com/keys](https://console.groq.com/keys)).

> LLM answers can change a little from run to run. That is normal.

## Step 0: Setup

We connect to Groq and write the same two tiny helpers as before:

- `make_agent(...)` creates an agent: a model plus a short job description.
- `ask(agent, text)` sends a message to an agent and gives back its reply as plain text.

`ask` also adds 1 to `calls["count"]` every time. We use this counter in Part 12 to measure cost.

In [1]:
import asyncio
import json
import os
import time
from collections import Counter
from dataclasses import dataclass

from dotenv import load_dotenv, find_dotenv
from agent_framework import WorkflowBuilder, WorkflowContext, executor, Case, Default
from agent_framework.openai import OpenAIChatCompletionClient
from openai import AsyncOpenAI

load_dotenv(find_dotenv(usecwd=True))

# One connection to Groq. Groq talks the same language as OpenAI.
# max_retries: if a call does not go through, wait a little and try again.
groq = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    async_client=AsyncOpenAI(
        api_key=os.getenv("GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1",
        max_retries=10,
    ),
)

# temperature 0 = the model gives its most likely answer every time (more stable)
# reasoning_effort low = less hidden thinking, so the model always gets to its answer
SETTINGS = {"temperature": 0, "reasoning_effort": "low"}

calls = {"count": 0}     # how many times ask() has used the model


def make_agent(name, instructions, tools=None):
    if tools:
        return groq.as_agent(name=name, instructions=instructions,
                             tools=tools, default_options=SETTINGS)
    return groq.as_agent(name=name, instructions=instructions,
                         default_options=SETTINGS)


async def ask(agent, text):
    calls["count"] += 1
    reply = await agent.run(text)
    return reply.text.strip()


print("Setup done")

Setup done


**Workflow reminder.** Three ideas are all you need for the graphs in this notebook:

1. **Executor**: one node in the graph. An `async` function with `@executor` on it.
2. **`ctx.send_message(x)`**: pass `x` along the edges to the next node or nodes.
3. **`ctx.yield_output(x)`**: "this is the final answer". Read it with `result.get_outputs()`.

`WorkflowBuilder` joins the nodes together with **edges**.

---
# Part 1: What is a multi-agent system?

A **multi-agent system** is a group of agents that work together on one problem.
Each agent has its own job, and something coordinates them.

Why not give one agent all the work? Let's try it first.

In [2]:
TICKET = "URGENT: someone hacked my account and I was charged twice"

generalist = make_agent(
    "generalist",
    "You are a support assistant. For the ticket, say if it is a security risk, "
    "how urgent it is, which team should take it, and write a reply to the customer.")

print(await ask(generalist, TICKET))

**Ticket Assessment**

| Item | Details |
|------|---------|
| **Security risk** | Yes – unauthorized access to the account and duplicate charge indicate a potential breach. |
| **Urgency** | **Critical** – the customer’s account has been compromised and funds have been taken. Immediate investigation and remediation are required. |
| **Responsible team** | **Security & Fraud Investigation Team** (with escalation to the Payment Operations team for the duplicate charge). |

---

**Customer Reply**

> **Subject:** Immediate Action on Your Account & Duplicate Charge  
> **From:** Support Team  
> **To:** [Customer Name]  
> **Date:** 2 Nov 2026  
> 
> Hi [Customer Name],
> 
> I’m really sorry to hear that your account was compromised and that you were charged twice. This is a top‑priority issue for us, and we’re already on it.
> 
> **What we’re doing right now:**
> 1. **Security team** has locked your account and is conducting a full audit of all recent activity.  
> 2. **Fraud team** is i

It gives an answer, but look at what we got: one long piece of text doing four jobs.
How would a program read the priority out of it? How would you test only the security check?

| | One big agent | A team of agents |
|---|---|---|
| Instructions | one long prompt doing many jobs | one short prompt per job |
| Testing | test everything at once | test each agent alone |
| When it goes wrong | hard to see which part failed | you can see which agent failed |
| Changing one job | might break the others | change one agent only |
| Cost | fewer model calls | more model calls |
| Coordination | none needed | **you must design it** |

That last row is the whole unit. Agents are easy. **Coordinating them is the real work.**

A team is not always better. For a small job, one agent is simpler and cheaper.
Use a team when the jobs are different, or when you need to check each one separately.

---
# Part 2: Role-based agents

A **role** is one clear job title: what this agent does and what it replies with.
We keep all the roles in one dictionary, so the team is easy to read and easy to grow.

In [3]:
ROLES = {
    "security":  "You are a security checker. Reply with one word: RISK if the text mentions "
                 "hacking or stolen passwords, otherwise OK.",
    "priority":  "You are a priority checker. Reply with one word: HIGH if the text is urgent, "
                 "otherwise NORMAL.",
    "sentiment": "You are a mood checker. Reply with one word: ANGRY if the customer is upset, "
                 "otherwise CALM.",
    "team":      "You are a support receptionist. Reply with one word, the team for this ticket: "
                 "billing (charges, refunds), technical (crashes, errors), "
                 "account (logins, being logged out, passwords), or general.",
    "summary":   "You are a summary writer. Write a one-sentence note using ONLY the "
                 "information you are given.",
}

# one line builds the whole team
team = {name: make_agent(name, job) for name, job in ROLES.items()}

print("team:", list(team))
print("security says:", await ask(team["security"], TICKET))
print("priority says:", await ask(team["priority"], TICKET))

team: ['security', 'priority', 'sentiment', 'team', 'summary']


security says: RISK


priority says: HIGH


Now every answer is **one word**, so a program can use it and a test can check it.

**Good roles:** one job, a fixed reply format, and nothing that belongs to another role.

---
# Part 3: Fan-out and fan-in (L1)

Our three checkers answer three separate questions. None needs another's answer.
So why should security wait for sentiment?

- **Fan-out** = one input goes to many workers, and they all run *at the same time*.
- **Fan-in** = many answers are gathered into one step.

## 3.1 Why fan out? The time saved

Pretend each reviewer takes 1 second. In a line, the total is the **sum**.
Together, the total is about as long as the **slowest** one.

In [4]:
async def pretend_review():
    await asyncio.sleep(1)


start_time = time.perf_counter()
for _ in range(3):
    await pretend_review()
print(f"line took     {time.perf_counter() - start_time:.1f}s")

start_time = time.perf_counter()
await asyncio.gather(pretend_review(), pretend_review(), pretend_review())
print(f"fan-out took  {time.perf_counter() - start_time:.1f}s")

line took     3.0s


fan-out took  1.0s


## 3.2 The fan-out / fan-in graph

Now with our real agents.

```
               +--> security  --+
 dispatch -----+--> priority  --+----> combine
   (fan out)   +--> sentiment --+   (fan in)
```

- `add_fan_out_edges(dispatch, REVIEWERS)`: dispatch's one message goes to **all** reviewers.
- `add_fan_in_edges(REVIEWERS, combine)`: combine gets **all** their answers, as one list.

In [5]:
@executor(id="dispatch")
async def dispatch(ticket: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(ticket)      # one message. The edges deliver it to everyone.


@executor(id="security")
async def security(ticket: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message("security: " + await ask(team["security"], ticket))


@executor(id="priority")
async def priority(ticket: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message("priority: " + await ask(team["priority"], ticket))


@executor(id="sentiment")
async def sentiment(ticket: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message("sentiment: " + await ask(team["sentiment"], ticket))


@executor(id="combine")
async def combine(findings: list[str], ctx: WorkflowContext) -> None:
    # notice list[str]: combine runs ONCE with ALL findings, not three times
    await ctx.yield_output(sorted(findings))


REVIEWERS = [security, priority, sentiment]

review_flow = (WorkflowBuilder(start_executor=dispatch)
               .add_fan_out_edges(dispatch, REVIEWERS)     # one -> many
               .add_fan_in_edges(REVIEWERS, combine)       # many -> one
               .build())

result = await review_flow.run("URGENT: someone tried to hack my account, this is terrible")
print(result.get_outputs()[0])

['priority: HIGH', 'security: RISK', 'sentiment: ANGRY']


Three reviewers, one list, and always exactly three findings. `combine` never sees a half-finished set.

## 3.3 Fan-in is a barrier

A **barrier** means "nobody continues until everyone has arrived".
Think of a school trip: the bus does not leave until the slowest kid is on board.
These workers are plain functions, so we can see the order clearly.

In [6]:
log = []


@executor(id="go")
async def go(x: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(x)


@executor(id="fast_kid")
async def fast_kid(x: str, ctx: WorkflowContext[str]) -> None:
    log.append("fast_kid boarded")
    await ctx.send_message("fast")


@executor(id="slow_kid")
async def slow_kid(x: str, ctx: WorkflowContext[str]) -> None:
    await asyncio.sleep(1)              # still at the gift shop
    log.append("slow_kid boarded")
    await ctx.send_message("slow")


@executor(id="bus_leaves")
async def bus_leaves(kids: list[str], ctx: WorkflowContext) -> None:
    log.append(f"BUS LEFT with {len(kids)} kids")
    await ctx.yield_output(kids)


trip = (WorkflowBuilder(start_executor=go)
        .add_fan_out_edges(go, [fast_kid, slow_kid])
        .add_fan_in_edges([fast_kid, slow_kid], bus_leaves)
        .build())

await trip.run("go")
print("\n".join(log))

fast_kid boarded
slow_kid boarded
BUS LEFT with 2 kids


The bus left **last**. You never wrote a "wait until both are done" check. The fan-in edge did it.

## 3.4 When NOT to fan out

Ask one question: **does step B need step A's answer?**

| Task pair | Independent? | Shape |
|---|---|---|
| security check + mood check | yes, both just read the ticket | **fan** |
| pick the team, then assign a person from that team | **no** | **line** |
| translate the ticket, then check its mood | **no** | **line** |

---
# Part 4: Routing (L2)

Fan-out sent the ticket to **everyone**. But a refund request should not go to the password team.
**Routing** = look at the work, then send it to exactly **one** handler.

```
                   +-- billing?   --> billing_desk
 classify --------+-- technical? --> technical_desk
  (decides)        +-- account?   --> account_desk
                   +-- otherwise  --> general_desk   (Default)
```

**Rule: decide in one place, act in another.** `classify` makes the decision. The edges only read it.

## 4.1 The router

In [7]:
@dataclass
class Routed:
    ticket: str
    queue: str      # the decision: billing / technical / account / general


def pick_queue(ticket):
    text = ticket.lower()
    if any(word in text for word in ("charge", "charged", "refund", "invoice", "payment")):
        return "billing"
    if any(word in text for word in ("error", "bug", "crash", "broken", "outage")):
        return "technical"
    if any(word in text for word in ("login", "password", "locked", "account")):
        return "account"
    return "general"


@executor(id="classify")
async def classify(ticket: str, ctx: WorkflowContext[Routed]) -> None:
    await ctx.send_message(Routed(ticket, pick_queue(ticket)))


@executor(id="billing_desk")
async def billing_desk(r: Routed, ctx: WorkflowContext) -> None:
    await ctx.yield_output("[BILLING]   " + r.ticket)


@executor(id="technical_desk")
async def technical_desk(r: Routed, ctx: WorkflowContext) -> None:
    await ctx.yield_output("[TECHNICAL] " + r.ticket)


@executor(id="account_desk")
async def account_desk(r: Routed, ctx: WorkflowContext) -> None:
    await ctx.yield_output("[ACCOUNT]   " + r.ticket)


@executor(id="general_desk")
async def general_desk(r: Routed, ctx: WorkflowContext) -> None:
    await ctx.yield_output("[GENERAL]   " + r.ticket)


router = (WorkflowBuilder(start_executor=classify)
          .add_switch_case_edge_group(classify, [
              Case(condition=lambda r: r.queue == "billing",   target=billing_desk),
              Case(condition=lambda r: r.queue == "technical", target=technical_desk),
              Case(condition=lambda r: r.queue == "account",   target=account_desk),
              Default(target=general_desk),
          ])
          .build())

for text in ["I was charged twice, need a refund",
             "the app crashes with an error",
             "I am locked out, forgot my password",
             "hello, just saying hi"]:
    result = await router.run(text)
    print(result.get_outputs()[0])

[BILLING]   I was charged twice, need a refund


[TECHNICAL] the app crashes with an error


[ACCOUNT]   I am locked out, forgot my password


[GENERAL]   hello, just saying hi


Each ticket went to **one** desk. The greeting matched nothing, so it went to the **Default**.

**Fan-out = all run. Routing = one runs.**

## 4.2 Two traps

**Trap 1: first match wins.** Cases are checked top to bottom, just like `if / elif`.
**Trap 2: the missing default.** Here is what each kind of routing does with a ticket that matches nothing.

In [8]:
# Trap 1: this ticket is about a charge AND an error. Billing is checked first.
result = await router.run("I was charged for a plan that gives an error")
print("Trap 1:", result.get_outputs()[0])

# Trap 2a: the switch-case refuses to build without a Default
try:
    (WorkflowBuilder(start_executor=classify)
     .add_switch_case_edge_group(classify, [
         Case(condition=lambda r: r.queue == "billing",   target=billing_desk),
         Case(condition=lambda r: r.queue == "technical", target=technical_desk),
     ])
     .build())
except ValueError as error:
    print("Trap 2a, refused to build:", error)

# Trap 2b: hand-made conditional edges give no such protection
leaky = (WorkflowBuilder(start_executor=classify)
         .add_edge(classify, billing_desk, condition=lambda r: r.queue == "billing")
         .build())

result = await leaky.run("hello, just saying hi")
print("Trap 2b, outputs:", result.get_outputs())

Trap 1: [BILLING]   I was charged for a plan that gives an error
Trap 2a, refused to build: SwitchCaseEdgeGroup must contain exactly one default case.


Trap 2b, outputs: []


The last list is empty. The customer's ticket is simply **gone**, with no error.
That is worse than a crash, because a crash at least gets noticed.

**Lessons:** put the most specific cases first, and always use the switch-case with a `Default`.

---
# Part 5: The manager and task delegation (L3)

Keyword rules cannot understand meaning. Look at the last ticket:

In [9]:
print(pick_queue("I was charged twice"))
print(pick_queue("The thing I paid for keeps logging me out"))

billing
general


The second ticket is really an **account** problem, but no keyword matched.

**L3's idea:** let an LLM be the manager. It reads the ticket, understands it, and **delegates** to the right worker.

- A **worker** is a normal agent with one small job.
- `as_tool(...)` wraps a worker so that another agent can call it, like calling a function.
- The **description** is how the manager knows what each worker does.

In [10]:
billing_agent = make_agent(
    "billing", "You are a billing specialist. You help with charges and refunds. Reply in one short sentence.")

technical_agent = make_agent(
    "technical", "You are a technical specialist. You help with crashes and errors. Reply in one short sentence.")

account_agent = make_agent(
    "account", "You are an account specialist. You help with logins and passwords. Reply in one short sentence.")

manager = make_agent(
    "manager",
    "You are a support manager. Never answer yourself. "
    "Pick the right specialist, call their tool, and return their answer.",
    tools=[
        billing_agent.as_tool(name="ask_billing",
                              description="Ask the billing specialist about charges and refunds."),
        technical_agent.as_tool(name="ask_technical",
                                description="Ask the technical specialist about crashes and errors."),
        account_agent.as_tool(name="ask_account",
                              description="Ask the account specialist about logins, being logged out, and passwords."),
    ],
)


def who_was_called(reply):
    names = []
    for message in reply.messages:
        for item in message.contents:
            if item.type == "function_call":
                names.append(item.name)
    return names


reply = await manager.run("The thing I paid for keeps logging me out")
print("Manager sent it to:", who_was_called(reply))
print("Answer:", reply.text)

Manager sent it to: ['ask_account']
Answer: It sounds like your subscription might not be active or there could be a session issue. Please check that your payment went through and your subscription is still active. If it is, try clearing your browser cache or resetting your password to see if that resolves the logout problem.


**Behind the scenes (about 3 model calls):** the manager picks a tool, the worker answers,
and the manager returns the answer.

**Lesson:** a tool description is an **instruction to the model**. With a lazy description,
nothing crashes. The customer just gets help from the wrong person.

| | Rules (L2) | Manager (L3) |
|---|---|---|
| Decides by | keywords you wrote | the LLM's understanding |
| Cost | 0 model calls | about 3 model calls per ticket |
| Messy tickets | often fails | usually gets it right |
| Same answer every time? | yes | not always |

Real systems often use **both**. We do exactly that in Part 12.

---
# Part 6: The blackboard (L4)

## 6.1 The problem: passing only messages

Each step passes one message to the next, like passing a parcel.
Each step sees only what the step before it gave it.

In [11]:
DEMO_TICKET = "URGENT: the app shows a blank screen and my client demo starts in 10 minutes"


@executor(id="msg_security")
async def msg_security(ticket: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message("security: " + await ask(team["security"], ticket))


@executor(id="msg_priority")
async def msg_priority(message: str, ctx: WorkflowContext) -> None:
    # PROBLEM: message is "security: OK", not the ticket!
    answer = await ask(team["priority"], message)
    await ctx.yield_output(f"priority saw: {message!r}\npriority said: {answer}")


message_flow = (WorkflowBuilder(start_executor=msg_security)
                .add_edge(msg_security, msg_priority)
                .build())

result = await message_flow.run(DEMO_TICKET)
print(result.get_outputs()[0])

priority saw: 'security: OK'
priority said: NORMAL


The priority agent never saw the word "URGENT". The ticket was lost after the first step.

## 6.2 The blackboard: one board everyone shares

```
        +------------ shared board ------------+
        | ticket: "URGENT: the app shows..."   |
        | notes:  ["security: OK",             |
        |          "priority: HIGH"]           |
        +--------------------------------------+
           ^           ^            ^          ^
        start  ->  security  ->  priority  ->  summary
```

- `ctx.set_state("name", value)` : write something on the board
- `ctx.get_state("name")` : read it back

**Every writer does 3 steps: read the notes, add yours, write them back.**

In [12]:
@executor(id="start")
async def start(ticket: str, ctx: WorkflowContext[str]) -> None:
    ctx.set_state("ticket", ticket)
    ctx.set_state("notes", [])
    await ctx.send_message("go")        # the message now just means "your turn"


@executor(id="board_security")
async def board_security(_: str, ctx: WorkflowContext[str]) -> None:
    ticket = ctx.get_state("ticket")
    notes = ctx.get_state("notes")                       # 1. READ
    answer = await ask(team["security"], ticket)
    ctx.set_state("notes", notes + ["security: " + answer])   # 2. ADD and 3. WRITE
    await ctx.send_message("go")


@executor(id="board_priority")
async def board_priority(_: str, ctx: WorkflowContext[str]) -> None:
    ticket = ctx.get_state("ticket")                     # reads the REAL ticket
    notes = ctx.get_state("notes")
    answer = await ask(team["priority"], ticket)
    ctx.set_state("notes", notes + ["priority: " + answer])
    await ctx.send_message("go")


@executor(id="board_summary")
async def board_summary(_: str, ctx: WorkflowContext) -> None:
    info = "Ticket: " + ctx.get_state("ticket") + "\nNotes: " + ", ".join(ctx.get_state("notes"))
    await ctx.yield_output(info + "\nSummary: " + await ask(team["summary"], info))


board_flow = (WorkflowBuilder(start_executor=start)
              .add_edge(start, board_security)
              .add_edge(board_security, board_priority)
              .add_edge(board_priority, board_summary)
              .build())

result = await board_flow.run(DEMO_TICKET)
print(result.get_outputs()[0])

Ticket: URGENT: the app shows a blank screen and my client demo starts in 10 minutes
Notes: security: OK, priority: HIGH
Summary: Ticket: URGENT: the app shows a blank screen and my client demo starts in 10 minutes; security: OK, priority: HIGH.


Now priority reads the real ticket, and summary sees the ticket **and** all the notes.

**Edges decide who goes next. The board holds the shared information.**

## 6.3 Common mistake: forgetting to READ first

This agent writes without reading, so it wipes out everyone else's notes.

In [13]:
@executor(id="clean_start")
async def clean_start(ticket: str, ctx: WorkflowContext[str]) -> None:
    ctx.set_state("notes", ["security: RISK"])      # a note that is already on the board
    await ctx.send_message("go")


@executor(id="buggy_priority")
async def buggy_priority(_: str, ctx: WorkflowContext) -> None:
    # BUG: we did not read the old notes. This REPLACES the whole list.
    ctx.set_state("notes", ["priority: HIGH"])
    await ctx.yield_output(ctx.get_state("notes"))


buggy_flow = (WorkflowBuilder(start_executor=clean_start)
              .add_edge(clean_start, buggy_priority)
              .build())

result = await buggy_flow.run("hacked!")
print(result.get_outputs()[0])

['priority: HIGH']


`security: RISK` is **gone**, and no error was shown.

Two more things to watch out for:

- **Typos in names.** `ctx.get_state("note")` (missing the "s") gives back `None` with no error.
- **Agents running at the same time.** If two agents read the list at the same moment,
  the one that writes last wins. Simple fix: give each agent **its own name** on the board,
  like `"security_note"` and `"priority_note"`. We do this in Part 13.

---
# Part 7: Inter-agent communication

You have now seen all three ways that agents talk to each other.

| Way | Picture | Who sees it | Command |
|---|---|---|---|
| **Message** | passing a parcel | only the next step | `ctx.send_message(x)` |
| **Tool call** | asking a colleague a question and waiting | only the caller | `agent.as_tool(...)` |
| **Shared board** | a whiteboard in the office | every step | `ctx.set_state(...)` / `ctx.get_state(...)` |

How to choose:

- One step hands a result to the next: **message**.
- One agent needs a quick answer from an expert: **tool call**.
- Many agents build one shared picture: **board**.

**Agree on a format.** Agents only understand each other if the content is agreed too.
That is why our checkers reply with one fixed word, and why the planner in Part 8 replies with JSON.

---
# Part 8: Planner and executor

The manager handles one request by picking a worker. But this ticket has **three** problems in it:

> "I was charged twice, the app crashes when I open my invoice, and I forgot my password."

Here we split the thinking from the doing:

- The **planner** reads the ticket and writes a to-do list. It solves nothing.
- The **executor** works through the list, one step at a time.

```
 planner  -->  checks the plan  -->  executor  --(more steps?)--+
                                        ^                       |
                                        +-----------------------+
                                        |
                                        +--(finished)--> report
```

The big benefit: the plan is **written down**. You can read it, check it, or reject it *before* any work happens.

## 8.1 The planner

In [14]:
WORKERS = {"billing": billing_agent, "technical": technical_agent, "account": account_agent}

planner = make_agent(
    "planner",
    "You are a planner for a support team. Break the ticket into steps. Do not solve anything. "
    "Reply with ONLY a JSON list and no markdown. Each item has two keys: "
    '"worker" (billing, technical, or account) and "task" (one short sentence).')

MULTI_TICKET = "I was charged twice, the app crashes when I open my invoice, and I forgot my password."

plan_text = await ask(planner, MULTI_TICKET)
print(plan_text)

[
  {"worker":"billing","task":"Verify duplicate charge and issue refund"},
  {"worker":"technical","task":"Investigate and fix app crash on invoice view"},
  {"worker":"account","task":"Reset user password and provide new login instructions"}
]


## 8.2 Never trust a plan blindly

The plan came from a model, so we check it with plain code before running anything.
Models sometimes wrap JSON in markdown fences, so we remove those first.

In [15]:
def check_plan(plan_text):
    clean_text = plan_text.replace("```json", "").replace("```", "")
    plan = json.loads(clean_text)           # stops here if it is not valid JSON
    if not plan:
        raise ValueError("the plan is empty")
    for step in plan:
        if step["worker"] not in WORKERS:
            raise ValueError("unknown worker: " + step["worker"])
    return plan


print("good plan has", len(check_plan(plan_text)), "steps")

# a bad plan is stopped before any work happens
try:
    check_plan('[{"worker": "legal", "task": "sue the customer"}]')
except ValueError as error:
    print("bad plan rejected:", error)

good plan has 3 steps
bad plan rejected: unknown worker: legal


## 8.3 The executor, as a graph

`make_plan` writes the plan on the board. `do_step` does one step, then sends itself
`"more"` or `"finished"`. A switch-case decides where to go. This is the first **loop** in our graphs.

You will see a `Self-loop detected` warning when this graph is built. It is a useful reminder: a node
pointing at itself could run forever. Ours cannot, because the plan has a fixed number of steps.

In [16]:
@executor(id="make_plan")
async def make_plan(ticket: str, ctx: WorkflowContext[str]) -> None:
    plan = check_plan(await ask(planner, ticket))
    ctx.set_state("plan", plan)
    ctx.set_state("done", [])
    await ctx.send_message("more")


@executor(id="do_step")
async def do_step(_: str, ctx: WorkflowContext[str]) -> None:
    plan = ctx.get_state("plan")
    done = ctx.get_state("done")
    step = plan[len(done)]                       # the next step not done yet

    answer = await ask(WORKERS[step["worker"]], step["task"])    # the worker sees only its own task

    done = done + [step["worker"] + ": " + answer]
    ctx.set_state("done", done)
    await ctx.send_message("more" if len(done) < len(plan) else "finished")


@executor(id="report")
async def report(_: str, ctx: WorkflowContext) -> None:
    await ctx.yield_output(ctx.get_state("done"))


plan_flow = (WorkflowBuilder(start_executor=make_plan)
             .add_edge(make_plan, do_step)
             .add_switch_case_edge_group(do_step, [
                 Case(condition=lambda message: message == "more", target=do_step),
                 Default(target=report),
             ])
             .build())

result = await plan_flow.run(MULTI_TICKET)
for line in result.get_outputs()[0]:
    print("-", line)

Self-loop detected: Executor 'do_step' connects to itself. This may cause infinite recursion if not properly handled with conditions.


- billing: I’ve identified the duplicate charge and processed a refund for you.
- technical: Check the invoice view’s data binding for null values and ensure the layout’s views are correctly referenced before accessing them.
- account: Sure—reset the password to “Temp1234”, then log in with your username and the new password, and change it immediately after signing in.


All three problems were handled, by three different workers, in the planned order.

**Careful:** these workers are only chat agents with no real tools, so they cannot truly refund or reset anything, and a reply may even invent details. A real system would give each worker tools, and check their results.

| | Manager (Part 5) | Planner and executor |
|---|---|---|
| The plan lives | in the model's head | written down, on the board |
| Can you check it first? | no | yes, with plain code |
| Who runs the work | the manager calls its tools | a plain executor follows the list |
| Best for | one clear request | a job with several steps |
| Trap | picks the wrong worker | a bad plan, so always check it |

---
# Part 9: Collaborative agents and coordination

So far, agents mostly worked **next to** each other. Now they work **with** each other:
one agent's answer changes what another agent does.

## 9.1 Writer and checker: a review loop

A writer drafts a reply. A checker reads it, and either approves it or says what to fix.
The writer tries again with that feedback.

```
 writer  -->  checker  --(fix it)--> back to writer
                 |
                 +--(approved or out of tries)--> publish
```

**Every loop needs a stop rule.** Otherwise two agents can argue forever. Ours stops after 3 tries.

In [17]:
writer_agent = make_agent(
    "writer", "You write short replies to customers. Reply with only the message to the customer.")

checker_agent = make_agent(
    "checker",
    "You check a customer reply. It must contain an apology, a promise to reply within 2 days, "
    "and be under 30 words. "
    "If all three are true, reply with one word: APPROVED. "
    "Otherwise reply with one short sentence saying what to fix.")

MAX_TRIES = 3


@executor(id="review_start")
async def review_start(ticket: str, ctx: WorkflowContext[str]) -> None:
    ctx.set_state("ticket", ticket)
    ctx.set_state("feedback", "")
    ctx.set_state("tries", 0)
    await ctx.send_message("write")


@executor(id="write_draft")
async def write_draft(_: str, ctx: WorkflowContext[str]) -> None:
    request = "Ticket: " + ctx.get_state("ticket")
    feedback = ctx.get_state("feedback")
    if feedback:
        request += "\nYour last draft: " + ctx.get_state("draft") + "\nFix this: " + feedback

    ctx.set_state("draft", await ask(writer_agent, request))
    ctx.set_state("tries", ctx.get_state("tries") + 1)
    await ctx.send_message("check")


@executor(id="check_draft")
async def check_draft(_: str, ctx: WorkflowContext[str]) -> None:
    verdict = await ask(checker_agent, ctx.get_state("draft"))
    tries = ctx.get_state("tries")
    print(f"try {tries}: {ctx.get_state('draft')}\n   checker: {verdict}\n")

    if "APPROVED" in verdict or tries >= MAX_TRIES:
        await ctx.send_message("stop")
    else:
        ctx.set_state("feedback", verdict)
        await ctx.send_message("again")


@executor(id="publish")
async def publish(_: str, ctx: WorkflowContext) -> None:
    await ctx.yield_output(ctx.get_state("draft"))


review_loop = (WorkflowBuilder(start_executor=review_start)
               .add_edge(review_start, write_draft)
               .add_edge(write_draft, check_draft)
               .add_switch_case_edge_group(check_draft, [
                   Case(condition=lambda message: message == "again", target=write_draft),
                   Default(target=publish),
               ])
               .build())

result = await review_loop.run("My refund has not arrived after two weeks.")
print("FINAL:", result.get_outputs()[0])

try 1: I’m sorry to hear that. I’ll check the status of your refund right away and get back to you with an update within the next 24 hours. Thank you for your patience.
   checker: Reduce the word count to under 30 words.



try 2: I’m sorry your refund hasn’t arrived. I’ll check its status and update you within 24 hours. Thank you for your patience.
   checker: APPROVED

FINAL: I’m sorry your refund hasn’t arrived. I’ll check its status and update you within 24 hours. Thank you for your patience.


## 9.2 Team vote

Sometimes a ticket is unclear, and no single agent should decide alone.
Ask several agents, each looking from its own role, and take the **majority**.
A vote is just a fan-out with a counting step at the fan-in.

In [18]:
VOTE_RULE = " Reply with one word, the team that should own this ticket: billing, technical, or account."

voters = {
    "finance":  make_agent("finance",  "You work in finance and care most about money." + VOTE_RULE),
    "engineer": make_agent("engineer", "You are a software engineer and care most about bugs." + VOTE_RULE),
    "support":  make_agent("support",  "You lead support and care most about customer accounts." + VOTE_RULE),
}


def make_voter(name):
    @executor(id="vote_" + name)
    async def voter(ticket: str, ctx: WorkflowContext[str]) -> None:
        answer = await ask(voters[name], ticket)
        await ctx.send_message(answer.lower().strip(" ."))
    return voter


@executor(id="open_vote")
async def open_vote(ticket: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(ticket)


@executor(id="count_votes")
async def count_votes(votes: list[str], ctx: WorkflowContext) -> None:
    await ctx.yield_output(Counter(votes).most_common())


VOTERS = [make_voter(name) for name in voters]

vote_flow = (WorkflowBuilder(start_executor=open_vote)
             .add_fan_out_edges(open_vote, VOTERS)
             .add_fan_in_edges(VOTERS, count_votes)
             .build())

result = await vote_flow.run("I paid for premium but I still cannot log in to use it")
print("votes, most popular first:", result.get_outputs()[0])

votes, most popular first: [('account', 3)]


Voters can disagree, and that is the point. If all three pick different teams there is no majority,
so a real system would pass that ticket to a person.

## 9.3 Every way of coordinating agents

Coordination means deciding **who works, when, and on what**. This table is the whole unit in one place.

| Mechanism | Who decides? | Part |
|---|---|---|
| **Barrier** (fan-in) | the graph: wait for everyone | 3 |
| **Routing** | rules you wrote | 4 |
| **Delegation** | a manager agent | 5 |
| **Shared board** | agents read what others wrote | 6 |
| **Plan** | a planner, checked by code | 8 |
| **Review loop** | a checker agent, with a stop rule | 9.1 |
| **Vote** | the majority | 9.2 |

---
# Part 10: Distributed problem solving

**The idea:** a problem is too big for one worker, so split it into pieces. Each worker solves
only its own piece and knows nothing about the others. Then the answers are joined together.

A teacher marking 90 exams gives 30 each to three helpers, then adds up the results.

```
                 +--> worker 0 (its own tickets) --+
 split the work -+--> worker 1 (its own tickets) --+--> merge into one report
                 +--> worker 2 (its own tickets) --+
```

Compare with Part 3. There, every worker read the **same** ticket. Here, each worker gets a **different piece**.

The workers share nothing except their messages. That is why this shape can move to many machines
with a queue between them. Here they all run on one machine, but the design is the same.

## 10.1 Split, solve, merge

In [19]:
TODAYS_TICKETS = [
    "I was charged twice", "the app crashes on start",
    "refund has not arrived", "error when saving",
    "forgot my password", "cannot log in on my phone",
]


def split_work(items, pieces):
    return [items[i::pieces] for i in range(pieces)]


print(split_work(TODAYS_TICKETS, 3))

[['I was charged twice', 'error when saving'], ['the app crashes on start', 'forgot my password'], ['refund has not arrived', 'cannot log in on my phone']]


In [20]:
summariser = make_agent(
    "summariser", "Summarise these support tickets in one short sentence.")

merger = make_agent(
    "merger",
    "You get one-line reports from several workers. Write a two-sentence report for the boss. "
    "If a worker FAILED, say that part of the work is missing.")


def make_worker(number, broken=False):
    @executor(id=f"worker_{number}")
    async def worker(_: str, ctx: WorkflowContext[str]) -> None:
        try:
            if broken:
                raise RuntimeError("machine went down")
            my_tickets = ctx.get_state("pieces")[number]       # only MY piece
            answer = await ask(summariser, "\n".join(my_tickets))
            await ctx.send_message(f"worker {number}: {answer}")
        except Exception as error:
            await ctx.send_message(f"worker {number}: FAILED ({error})")
    return worker


@executor(id="split_step")
async def split_step(tickets: list[str], ctx: WorkflowContext[str]) -> None:
    ctx.set_state("pieces", split_work(tickets, 3))
    await ctx.send_message("go")


@executor(id="merge_step")
async def merge_step(reports: list[str], ctx: WorkflowContext) -> None:
    joined = "\n".join(sorted(reports))
    await ctx.yield_output(joined + "\n\nREPORT: " + await ask(merger, joined))


def build_split_flow(broken_worker=None):
    workers = [make_worker(i, broken=(i == broken_worker)) for i in range(3)]
    return (WorkflowBuilder(start_executor=split_step)
            .add_fan_out_edges(split_step, workers)
            .add_fan_in_edges(workers, merge_step)
            .build())


result = await build_split_flow().run(TODAYS_TICKETS)
print(result.get_outputs()[0])

worker 0: A user reported a duplicate charge and an error that prevented them from saving their information.
worker 1: App crashes on launch and user has forgotten their password.
worker 2: Both issues involve account access: the refund hasn’t been received and the user can’t log in on their phone.

REPORT: The team has identified three critical issues: a duplicate charge and a data‑saving error (worker 0), an app crash on launch coupled with a forgotten password (worker 1), and a missing refund plus login failure on mobile (worker 2). The missing refund and login problems are still unresolved and require immediate attention.


## 10.2 What if one worker fails?

With many workers, one failing is normal. Worker 1 goes down below. Because it catches the error
and still sends a message, the barrier still gets all three answers, and the report says what is missing.

Without that `try / except`, the error would stop the whole run and we would lose the other two answers.

In [21]:
result = await build_split_flow(broken_worker=1).run(TODAYS_TICKETS)
print(result.get_outputs()[0])

worker 0: A user reported a duplicate charge and an error that prevented them from saving their information.
worker 1: FAILED (machine went down)
worker 2: Both issues involve account access: the refund hasn’t been received and the user can’t log in on their phone.

REPORT: The team has identified two key customer issues: a duplicate charge and a failure to save user information, and a missing refund coupled with login problems on mobile devices. Unfortunately, the report from worker 1 is incomplete because the machine went down, so that portion of the work is currently missing.


---
# Part 11: Scaling

What happens when 1 ticket becomes thousands?

## 11.1 The service has limits

Every model service can only handle so much at once. Think of a restaurant with 5 tables:
if 20 groups walk in together, 15 are turned away.

`BusyService` below is a pretend model service. No real model is used here, so we can test heavy
load safely. It handles 5 requests at a time, and each one takes 0.1 seconds.

In [22]:
class BusyService:
    def __init__(self, capacity):
        self.capacity = capacity
        self.running = 0
        self.served = 0
        self.rejected = 0

    async def call(self):
        if self.running >= self.capacity:      # all tables are full
            self.rejected += 1
            return
        self.running += 1
        await asyncio.sleep(0.1)               # pretend the model is thinking
        self.running -= 1
        self.served += 1


def show(label, service, seconds):
    print(f"{label:16} {seconds:5.1f}s   served {service.served:2}   rejected {service.rejected:2}")


TICKET_COUNT = 20

## 11.2 Three ways to handle 20 tickets

1. **One by one**: safe, but slow.
2. **All at once**: fast, but most get rejected.
3. **With a limit**: an `asyncio.Semaphore` is a doorman who lets in only 5 at a time. The rest wait outside.

In [23]:
# 1. one by one
service = BusyService(capacity=5)
start_time = time.perf_counter()
for _ in range(TICKET_COUNT):
    await service.call()
show("one by one", service, time.perf_counter() - start_time)

# 2. all at once
service = BusyService(capacity=5)
start_time = time.perf_counter()
await asyncio.gather(*[service.call() for _ in range(TICKET_COUNT)])
show("all at once", service, time.perf_counter() - start_time)

# 3. all at once, with a limit of 5
service = BusyService(capacity=5)
doorman = asyncio.Semaphore(5)


async def call_with_limit():
    async with doorman:             # wait here until there is room
        await service.call()


start_time = time.perf_counter()
await asyncio.gather(*[call_with_limit() for _ in range(TICKET_COUNT)])
show("limit of 5", service, time.perf_counter() - start_time)

one by one         2.1s   served 20   rejected  0
all at once        0.1s   served  5   rejected 15


limit of 5         0.4s   served 20   rejected  0


With the limit, all 20 tickets are served, about 5 times faster than one by one, and none are rejected.
**More concurrency is only better up to what the service can take.**

## 11.3 Strategies for scaling collaboration

| Strategy | What it means | Where you saw it |
|---|---|---|
| **Limit concurrency** | only N calls at once, the rest wait | 11.2 |
| **Keep workers independent** | a worker needs only its input, so you can run more copies | 10 |
| **Use a queue** | put tickets in a line, and workers take the next one when free | the shape in 10 |
| **Add roles, not rewrites** | a new reviewer means one new item in a list | 3 (`REVIEWERS`) |
| **Plan for failure** | one worker failing must not stop the rest | 10.2 |
| **Spend model calls carefully** | use rules first, and the LLM only when needed | 12 |

## 11.4 Cost grows with the tickets

Each extra agent adds model calls to **every** ticket. A small design choice becomes a big bill at scale.

In [24]:
designs = {"rules only": 0, "one LLM classifier": 1, "three reviewers": 3, "manager": 3, "review loop, 3 tries": 6}

print(f"{'design':22}{'per ticket':>12}{'10,000 tickets':>16}")
for design, per_ticket in designs.items():
    print(f"{design:22}{per_ticket:>12}{per_ticket * 10_000:>16,}")

design                  per ticket  10,000 tickets
rules only                       0               0
one LLM classifier               1          10,000
three reviewers                  3          30,000
manager                          3          30,000
review loop, 3 tries             6          60,000


---
# Part 12: Workflow optimisation

Do not guess what is slow or expensive. **Measure it, fix the biggest cost, then measure again.**

`calls["count"]` is our measuring tool.

## 12.1 Use rules first, and the LLM only when needed

Most tickets are easy, so rules can handle them for free. We call the LLM only when rules have no answer.

In [25]:
QUEUES = ("billing", "technical", "account", "general")


async def llm_team(ticket):
    word = (await ask(team["team"], ticket)).lower().strip(" .")
    return word if word in QUEUES else "general"      # never trust the model blindly


async def decide_team(ticket):
    queue = pick_queue(ticket)                        # free
    if queue != "general":
        return queue
    return await llm_team(ticket)                     # only for the hard ones


SAMPLE = [
    "I was charged twice", "the app crashes on start", "forgot my password",
    "refund please", "error when saving", "The thing I paid for keeps logging me out",
]

before = calls["count"]
print("LLM for every ticket:", [await llm_team(t) for t in SAMPLE])
print("model calls:", calls["count"] - before)

before = calls["count"]
print("\nrules first:         ", [await decide_team(t) for t in SAMPLE])
print("model calls:", calls["count"] - before)

LLM for every ticket: ['billing', 'technical', 'account', 'billing', 'technical', 'account']
model calls: 6



rules first:          ['billing', 'technical', 'account', 'billing', 'technical', 'account']
model calls: 1


## 12.2 Do not ask the same question twice

The same ticket text often comes up again. A **cache** remembers the answer.

In [26]:
cache = {}


async def cached_team(ticket):
    key = ticket.strip().lower()
    if key not in cache:
        cache[key] = await decide_team(ticket)
    return cache[key]


before = calls["count"]
for text in ["My stuff keeps vanishing", "my stuff keeps vanishing", "MY STUFF KEEPS VANISHING "]:
    print(text, "->", await cached_team(text))

print("model calls for 3 tickets:", calls["count"] - before)

My stuff keeps vanishing -> account
my stuff keeps vanishing -> account
MY STUFF KEEPS VANISHING  -> account
model calls for 3 tickets: 1


## 12.3 Your optimisation checklist

| Lever | What to do | Where you saw it |
|---|---|---|
| **Run independent work in parallel** | fan out, so you wait for the slowest and not the sum | 3 |
| **Use rules first** | the LLM only for hard cases | 12.1 |
| **Cache** | never pay twice for the same question | 12.2 |
| **Stop early** | a loop needs a stop rule; skip work you do not need | 9.1 |
| **Send less text** | give each agent only what it needs | 8 and 10 |
| **Pick the simplest shape** | do not fan out or add a manager just because you can | 1 and 3 |

Fix the **biggest** cost first. Making the cheapest step faster changes nothing.

---
# Part 13: The full enterprise workflow

Now we join everything into one **collaborative enterprise workflow**.
"Enterprise" means a business depends on it, so it needs three extra things:

- **Clear ownership**: every ticket has one team responsible for it.
- **A human in charge of risk**: dangerous tickets go to a person, not to an automatic reply.
- **A record**: afterwards, you can see what was decided and why.

```
                 +-- billing?   --> billing_owner   --+
 intake --------+-- technical? --> technical_owner --+                 +--> check_security --+
 (rules, then    +-- account?   --> account_owner   --+--> start_checks +                     +--> decide
  LLM if needed) +-- otherwise  --> general_owner   --+                 +--> check_priority --+
```

| Step | Idea used | Part |
|---|---|---|
| Each agent has one job | role-based agents | 2 |
| `intake` uses rules first, then the LLM | optimisation | 12 |
| Switch-case with a Default | routing | 4 |
| Two checks at once, then a barrier | fan-out and fan-in | 3 |
| Each check writes its own note | blackboard | 6 |
| The right specialist writes the reply | delegation | 5 |
| Risky tickets stop for a human | collaboration with people | 13 |

In [27]:
SLA_HOURS = {"HIGH": 1, "NORMAL": 24}     # how fast the business promises to reply


@executor(id="intake")
async def intake(ticket: str, ctx: WorkflowContext[Routed]) -> None:
    ctx.set_state("ticket", ticket)
    await ctx.send_message(Routed(ticket, await decide_team(ticket)))


def make_owner(team_name):
    @executor(id=team_name + "_owner")
    async def owner(r: Routed, ctx: WorkflowContext[str]) -> None:
        ctx.set_state("owner", team_name)       # clear ownership, on the board
        await ctx.send_message("go")
    return owner


@executor(id="start_checks")
async def start_checks(_: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message("go")


# these two run at the same time, so each one has ITS OWN name on the board
@executor(id="check_security")
async def check_security(_: str, ctx: WorkflowContext[str]) -> None:
    ctx.set_state("security_note", await ask(team["security"], ctx.get_state("ticket")))
    await ctx.send_message("done")


@executor(id="check_priority")
async def check_priority(_: str, ctx: WorkflowContext[str]) -> None:
    ctx.set_state("priority_note", await ask(team["priority"], ctx.get_state("ticket")))
    await ctx.send_message("done")


@executor(id="decide")
async def decide(_: list[str], ctx: WorkflowContext) -> None:
    ticket = ctx.get_state("ticket")
    owner = ctx.get_state("owner")
    security_note = ctx.get_state("security_note")
    priority_note = ctx.get_state("priority_note")

    if "RISK" in security_note:
        action = "ESCALATE to a human on the security team"
        reply = "(none: a person must reply)"
    elif owner in WORKERS:
        action = "automatic reply from " + owner
        reply = await ask(WORKERS[owner], ticket)
    else:
        action = "send to the general support queue"
        reply = "(none: a person will read it)"

    # the record: what was decided, and everything it was based on
    await ctx.yield_output({
        "ticket": ticket,
        "owner": owner,
        "security": security_note,
        "priority": priority_note,
        "reply within (hours)": SLA_HOURS.get(priority_note, 24),
        "action": action,
        "reply": reply,
    })


billing_owner, technical_owner, account_owner, general_owner = [
    make_owner(name) for name in QUEUES]

OWNERS = [billing_owner, technical_owner, account_owner, general_owner]
CHECKS = [check_security, check_priority]

builder = (WorkflowBuilder(start_executor=intake)
           .add_switch_case_edge_group(intake, [
               Case(condition=lambda r: r.queue == "billing",   target=billing_owner),
               Case(condition=lambda r: r.queue == "technical", target=technical_owner),
               Case(condition=lambda r: r.queue == "account",   target=account_owner),
               Default(target=general_owner),
           ]))

for owner in OWNERS:
    builder = builder.add_edge(owner, start_checks)

support_system = (builder
                  .add_fan_out_edges(start_checks, CHECKS)
                  .add_fan_in_edges(CHECKS, decide)
                  .build())

print("full system built")

full system built


## 13.1 Three tickets through the whole system

- a normal billing ticket: gets an automatic reply
- an account hack: must stop for a human
- a messy ticket with no keywords: rules cannot place it, so the LLM does

In [28]:
for text in ["I was charged twice, please refund me",
             "URGENT: someone hacked my account and changed my password!",
             "The thing I paid for keeps logging me out"]:
    result = await support_system.run(text)
    for key, value in result.get_outputs()[0].items():
        print(f"{key:22}: {value}")
    print()

ticket                : I was charged twice, please refund me
owner                 : billing
security              : OK
priority              : NORMAL
reply within (hours)  : 24
action                : automatic reply from billing
reply                 : Sure, I’ll process a refund for the duplicate charge right away.



ticket                : URGENT: someone hacked my account and changed my password!
owner                 : account
security              : RISK
priority              : HIGH
reply within (hours)  : 1
action                : ESCALATE to a human on the security team
reply                 : (none: a person must reply)



ticket                : The thing I paid for keeps logging me out
owner                 : account
security              : OK
priority              : NORMAL
reply within (hours)  : 24
action                : automatic reply from account
reply                 : Try resetting your password and make sure “Remember me” is checked.



Read each record and you can follow the whole story: who owned the ticket, what each check found,
what was promised, and what happened. The hacked account got **no** automatic reply, because
a business does not let a model handle that alone.

**The board is what joins it all together.** Intake, the owner and the checks each wrote one part,
and `decide` read all of it.

---
# Cheat sheet

| Pattern | Big idea | Key code | Common mistake |
|---|---|---|---|
| **Roles** | one agent, one job | `ROLES`, `make_agent` | one agent doing many jobs |
| **Fan-out / fan-in** | all run, then join | `add_fan_out_edges`, `add_fan_in_edges` | fanning out work that depends on each other |
| **Routing** | exactly one runs | `add_switch_case_edge_group`, `Case`, `Default` | no default, so tickets vanish |
| **Manager** | an LLM picks the worker | `agent.as_tool(name, description)` | lazy tool descriptions |
| **Blackboard** | one shared board | `ctx.set_state`, `ctx.get_state` | writing without reading first |
| **Planner and executor** | write the plan, then follow it | `check_plan`, a switch-case loop | running a plan without checking it |
| **Review loop** | one agent improves another's work | a switch-case loop, `MAX_TRIES` | no stop rule |
| **Vote** | the majority decides | fan-out plus `Counter` | an even number of voters can tie |
| **Split and merge** | each worker gets one piece | `split_work`, `make_worker` | one failure stops everything |
| **Limit concurrency** | only N at once | `asyncio.Semaphore` | sending everything at once |
| **Optimise** | measure, then fix the biggest cost | `calls["count"]`, `cache` | guessing what is slow |

## The whole syllabus, and where to find it

| Syllabus topic | Part |
|---|---|
| Introduction to multi-agent systems | 1 |
| Planner-executor architectures | 8 |
| Collaborative agents | 9 |
| Inter-agent communication | 7 |
| Task delegation | 5, 8 |
| Distributed problem solving | 10 |
| Orchestration patterns | 3, 4, 5, 6, 8 |
| Agent coordination mechanisms | 9.3 |
| Collaborative enterprise workflows | 13 |
| Role-based agents | 2 |
| Scalable AI collaboration strategies | 11 |
| Workflow optimisation for multi-agent systems | 12 |

## Try it yourself

1. Add a `spam` role to `ROLES` and to `REVIEWERS` in Part 3. What did you have to change?
2. Add a `sales` worker to the planner in Part 8. Update `WORKERS` and the planner's instructions, then send a ticket asking for a discount.
3. In Part 9.1, set `MAX_TRIES = 1`. What gets published?
4. In Part 11.2, change the semaphore to 10. How many tickets get rejected, and why?
5. In Part 13, add a `sentiment` check to `CHECKS`. Remember to give it its own name on the board, and show it in the record.